In [1]:
from pynq.overlays.base import BaseOverlay
import time
import asyncio
from pynq.lib import Pmod_ADC
import struct

base = BaseOverlay("base.bit")

In [2]:
%load_ext autoreload
%autoreload 2
from pmod_ir_transceiver import Pmod_IRTransceiver

Sensor Module Code Blocks [Temp, HeartBeat, Brightness]

In [3]:
import math as mt

def thermistorTemp(Vout):
    # Voltage Divider
    Vin = 5;
    Ro = 10000 # 10k Resistor
    # Steinhart Constants
    A = 0.001129148
    B = 0.000234125
    C = 0.0000000876741
    # Calculate Resistance
    Rt = (Vout * Ro) / (Vin - Vout)
    #Rt = 10000 # Used for Testing. Setting Rt=10k should give TempC=25
    # Steinhart - Hart Equation
    TempK = 1 / (A + (B * mt.log(Rt)) + C * mt.pow(mt.log(Rt),3))
    # Convert from Kelvin to Celsius
    TempC = TempK - 273.15
    return TempC

In [31]:
##heartbeat monitor!
import time

class BPMEstimator:
    def __init__(self,
                 threshold=0.6,
                 refractory=0.30,   # seconds
                 alpha=0.2):        # smoothing

        self.threshold = threshold
        self.refractory = refractory
        self.alpha = alpha

        self.last_value = 0
        self.last_beat_time = None
        self.last_detect_time = 0

        self.bpm = 70

    def update(self, value):
        """
        Call this once per sample.
        value = normalized signal (0–1 preferred)
        """
        now = time.monotonic() * 1000

        # Rising edge threshold crossing
        beat_detected = (
            self.last_value < self.threshold and
            value >= self.threshold and
            (now - self.last_detect_time) > self.refractory
        )

        if beat_detected:
            self.last_detect_time = now

            if self.last_beat_time is not None:
                dt = now - self.last_beat_time
                instant_bpm = 60.0 / dt
                # EMA smoothing
                if self.bpm is None:
                    self.bpm = instant_bpm
                else:
                    self.bpm = (
                        self.alpha * instant_bpm +
                        (1 - self.alpha) * self.bpm
                    )

            self.last_beat_time = now

        self.last_value = value
#         print(self.bpm)
        return self.bpm

In [ ]:

bpm = 70
adc = Pmod_ADC(base.PMODB)
state = 'SENSOR'
packet_size = 20
start = True
sleep_time = 0.5
ir_tx = Pmod_IRTransceiver(base.PMODA, 0, 1) # def __init__(self, mb_info, send_idx, receive_idx):

btns = base.btns_gpio

async def get_btns(_loop):
    global start,state
    while start:
        await asyncio.sleep(0.01)
        button = btns.read()
        if button & 0x1:
            state = 'STRING'
        elif button & 0x2:
            state = 'SENSOR'
       
        elif button & 0x8:
            _loop.stop()
            start = False

            
def packetize(packet_type,msg):
    #need to turn string to bytes 
    #first byte as packet type for receiver! 
    #next bytes at payload to 19 bytes, 
    #always pad to 19 bytes
         
    packet = bytearray(packet_size)

    if packet_type == 'str':
        payload = msg.encode()
        length = len(payload)
   
        packet[0] = 1 # packet type 1 = string 
        packet[1] = 3 #line item
    
    elif packet_type == 'sensor_t':
        packet[0] = 2 # packet type 2 = temp_sensor
        packet[1] = 1  ##  1 for temp data? 
        float_bytes = struct.pack('f',msg)
        length = len(float_bytes)
        payload = float_bytes
    
    elif packet_type == 'sensor_hb':
        packet[0] = 2 # packet type 2 = temp_sensor
        packet[1] = 3  ## 3 for heartbeat? 
        
        float_bytes = struct.pack('f',msg)
        length = len(float_bytes)
        payload = float_bytes
    elif packet_type == 'sensor_l':
        packet[0] = 2 # packet type 2 = temp_sensor
        packet[1] = 2  ## 3 for heartbeat? 
        
        float_bytes = struct.pack('f',msg)
        length = len(float_bytes)
        payload = float_bytes
    
    packet[2:2+length] = payload   
#     print(packet)
    return packet
    
def read_temp_data():
    voltage = adc.read(ch1=1, ch2=0, ch3=1)[0]
#     print(voltage)
    temp_c = thermistorTemp(voltage)
    temp_f = ((temp_c - 20) * 1.8) + 32
    return temp_f

def read_light_data():
    voltage = adc.read(ch1=1, ch2=0, ch3=1)[1]   
#     print(voltage)
    return voltage


async def tx_loop():
    global bpm,state,start
    while start:
        # wait to make sure read starts first
        await asyncio.sleep(0.5)
        
        if state == 'SENSOR':
        
            temp_f = read_temp_data()
#             print(temp_f)
            full_packet = list(packetize('sensor_t',temp_f))
            await ir_tx.write_async(full_packet)

            await asyncio.sleep(sleep_time)

#             hb_packet = list(packetize('sensor_hb',bpm))
#             await ir_tx.write_async(hb_packet)
#             await asyncio.sleep(0.05)

            light = read_light_data()
            light_packet = list(packetize('sensor_l',light))
            await ir_tx.write_async(light_packet)
            await asyncio.sleep(sleep_time)

        elif state == 'STRING':
     
            string = input("Input a string to send!")
            if len(string) > 16:
                print("string is too long, skipping......")
                continue
                
            full_packet = list(packetize('str',string))

            for i in range(3):
                await ir_tx.write_async(full_packet)
                await asyncio.sleep(sleep_time)

        
async def heartbeat_loop():
    estimator = BPMEstimator()
    global bpm,start
    while start:   
        sample = adc.read(ch1=0, ch2=1, ch3=0)[0]
        
        bpm = estimator.update(sample)
#         if bpm: 
#             print(f"BPM: {bpm:.1f}")
        await asyncio.sleep(0.05)

loop = asyncio.new_event_loop()
# loop.create_task(heartbeat_loop())
loop.create_task(get_btns(loop))
loop.create_task(tx_loop())
loop.run_forever()
loop.close()
print("Done.")

Task was destroyed but it is pending!
task: <Task pending name='Task-8' coro=<tx_loop() done, defined at /tmp/ipykernel_915/542319918.py:80> wait_for=<Future pending cb=[Task.__wakeup()]>>


sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 bytes
sending 20 byt